In [1]:
import sys
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from schemas.movie import Movie
from movie_ingestion.final_ingestion.vector_text import create_production_vector_text

# ============================================================
#  Set test IDs here
# ============================================================

tmdb_ids = [
    11,         # Star Wars (original, franchise starter, big budget for era)
    12,         # Finding Nemo (animation, computer animation)
    2,          # Ariel (animation)
    8,          # Marley & Me (live action, drama)
    3,          # Grumpier Old Men (live action, comedy)
    5,          # Four Rooms (live action, low budget)
    219,        # ESB (sequel, franchise continuation)
    58,         # Pirates of the Caribbean: Dead Man's Chest (CGI in live-action)
    1893,       # Star Wars: Episode I (CGI in live-action, big budget)
    299534,     # Avengers: Endgame (big budget blockbuster, franchise)
    10137,      # Stuart Little (live-action/CGI hybrid)
    1643729, 1611977, 1604474,  # misc test IDs
]

# ============================================================

SEPARATOR = "=" * 80

for tmdb_id in tmdb_ids:
    movie = Movie.from_tmdb_id(tmdb_id)
    title = movie.imdb_data.original_title or movie.tmdb_data.title

    print(SEPARATOR)
    print(f"  [{tmdb_id}] {title}")
    print(SEPARATOR)

    # --- Raw structured data ---
    print(f"\n--- IMDBData fields ---")
    print(f"  countries_of_origin: {movie.imdb_data.countries_of_origin}")
    print(f"  production_companies: {movie.imdb_data.production_companies}")
    print(f"  filming_locations: {movie.imdb_data.filming_locations}")
    print(f"  languages: {movie.imdb_data.languages}")
    print(f"  genres: {movie.imdb_data.genres}")
    print(f"  overall_keywords: {movie.imdb_data.overall_keywords}")
    print(f"  plot_keywords: {movie.imdb_data.plot_keywords}")

    print(f"\n--- TMDBData fields ---")
    print(f"  release_date: {movie.tmdb_data.release_date}")
    print(f"  budget (tmdb): {movie.tmdb_data.budget}")
    print(f"  budget (imdb): {movie.imdb_data.budget}")
    print(f"  resolved_budget(): {movie.resolved_budget()}")

    # --- Derived methods ---
    print(f"\n--- Derived production fields ---")
    print(f"  is_animation(): {movie.is_animation()}")
    print(f"  production_text(locations={not movie.is_animation()}): {movie.production_text(include_filming_locations=not movie.is_animation())}")
    print(f"  languages_text(): {movie.languages_text()}")
    print(f"  release_decade_bucket(): {movie.release_decade_bucket()}")
    print(f"  budget_bucket_for_era(): {movie.budget_bucket_for_era()}")

    # --- LLM metadata ---
    print(f"\n--- ProductionKeywordsOutput ---")
    pk = movie.production_keywords_metadata
    if pk:
        print(f"  terms: {pk.terms}")
        print(f"  embedding_text(): {pk.embedding_text()}")
    else:
        print("  (none)")

    print(f"\n--- SourceOfInspirationOutput ---")
    soi = movie.source_of_inspiration_metadata
    if soi:
        print(f"  source_material: {soi.source_material}")
        print(f"  franchise_lineage: {soi.franchise_lineage}")
        print(f"  embedding_text(): {soi.embedding_text()}")
    else:
        print("  (none)")

    # --- Final vector text ---
    print(f"\n--- create_production_vector_text() ---")
    print(create_production_vector_text(movie))
    print()

  [11] Star Wars

--- IMDBData fields ---
  countries_of_origin: ['United States']
  production_companies: ['Lucasfilm', 'Twentieth Century Fox']
  filming_locations: ['Tikal National Park, Guatemala', 'Sidi Driss Hotel, Matmata, Tunisia', 'Chott el Djerid, Nefta, Tunisia', 'Sidi Bouhlel, Tozeur, Tunisia', 'Ajim, Jerba, Tunisia', 'Yuma, Arizona, USA', 'Calakmul, Campeche, Mexico', 'Sidi Bouhel, Tunisia', 'Death Valley National Park, California, USA', 'Elstree Studios, Borehamwood, Hertfordshire, England, UK']
  languages: ['English']
  genres: ['Action', 'Adventure', 'Fantasy', 'Sci-Fi']
  overall_keywords: ['Action Epic', 'Adventure Epic', 'Dystopian Sci-Fi', 'Epic', 'Fantasy Epic', 'Quest', 'Sci-Fi Epic', 'Space Sci-Fi', 'Sword & Sorcery', 'Action', 'Adventure', 'Fantasy', 'Sci-Fi']
  plot_keywords: ['rebellion', 'princess', 'space opera', 'good versus evil', 'lightsaber', 'droid', 'jedi', 'death star', 'space station', 'weapon of mass destruction']

--- TMDBData fields ---
  release